# 01 — Data Quality Check: `sets.csv` & `themes.csv`

Rebrickable LEGO veri setinin temel doğrulaması. Amaç:

1. Şema kontrolü (satır sayısı, kolonlar, dtype'lar)
2. Null oranları
3. Yıl aralığı kontrolü
4. `sets.theme_id` → `themes.id` join kontrolü (yetim kayıtlar)
5. `num_parts = 0` olan setlerin oranı (muhtemelen minifig-only / promosyon setleri — ana analizden filtrelenecek)


In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

RAW = "../data/raw"


## 1. Veriyi oku

In [2]:
sets = pd.read_csv(f"{RAW}/sets.csv")
themes = pd.read_csv(f"{RAW}/themes.csv")

print(f"sets:   {sets.shape[0]:,} satır, {sets.shape[1]} kolon")
print(f"themes: {themes.shape[0]:,} satır, {themes.shape[1]} kolon")


sets:   28,180 satır, 6 kolon
themes: 496 satır, 3 kolon


## 2. Şema kontrolü

In [3]:
print("--- sets.csv ---")
sets.info()


--- sets.csv ---
<class 'pandas.DataFrame'>
RangeIndex: 28180 entries, 0 to 28179
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   set_num    28180 non-null  str  
 1   name       28180 non-null  str  
 2   year       28180 non-null  int64
 3   theme_id   28180 non-null  int64
 4   num_parts  28180 non-null  int64
 5   img_url    28180 non-null  str  
dtypes: int64(3), str(3)
memory usage: 1.3 MB


In [4]:
sets.head()


,set_num,name,year,theme_id,num_parts,img_url
0,0003977811-1,Ninjago: Book of Adventures,2022,761,1,https://cdn.rebrickable.com/media/sets/0003977...
1,001-1,Gears,1965,756,43,https://cdn.rebrickable.com/media/sets/001-1.jpg
2,0011-2,Town Mini-Figures,1979,67,12,https://cdn.rebrickable.com/media/sets/0011-2.jpg
3,0011-3,Castle 2 for 1 Bonus Offer,1987,199,0,https://cdn.rebrickable.com/media/sets/0011-3.jpg
4,0012-1,Space Mini-Figures,1979,143,12,https://cdn.rebrickable.com/media/sets/0012-1.jpg


In [5]:
print("--- themes.csv ---")
themes.info()


--- themes.csv ---
<class 'pandas.DataFrame'>
RangeIndex: 496 entries, 0 to 495
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         496 non-null    int64  
 1   name       496 non-null    str    
 2   parent_id  346 non-null    float64
dtypes: float64(1), int64(1), str(1)
memory usage: 11.8 KB


In [6]:
themes.head()


,id,name,parent_id
0,1,Technic,NaN
1,3,Competition,1.0
2,4,Expert Builder,1.0
3,16,RoboRiders,1.0
4,17,Speed Slammers,1.0


## 3. Null oranları

In [7]:
def null_report(df, name):
    rep = pd.DataFrame({
        "null_count": df.isna().sum(),
        "null_ratio_%": (df.isna().mean() * 100).round(2),
    })
    rep.index.name = f"{name} column"
    return rep

null_report(sets, "sets")


,null_count,null_ratio_%
sets column,,
set_num,0,0.0
name,0,0.0
year,0,0.0
theme_id,0,0.0
num_parts,0,0.0
img_url,0,0.0


In [8]:
null_report(themes, "themes")


,null_count,null_ratio_%
themes column,,
id,0,0.00
name,0,0.00
parent_id,150,30.24


## 4. Yıl aralığı kontrolü

`sets.year` üzerinde min/max, dağılım ve makul olmayan (örn. gelecek yıl veya çok eski / 0) değer kontrolü.


In [9]:
from datetime import datetime

current_year = datetime.now().year

print(f"year min : {sets['year'].min()}")
print(f"year max : {sets['year'].max()}")
print(f"unique yıl sayısı: {sets['year'].nunique()}")
print()

future_years = sets[sets["year"] > current_year]
print(f"Bugünün yılından ({current_year}) büyük yıl içeren set sayısı: {len(future_years)}")
if len(future_years):
    display(future_years[["set_num", "name", "year"]].sort_values("year", ascending=False).head(10))


year min : 1949
year max : 2027
unique yıl sayısı: 77

Bugünün yılından (2026) büyük yıl içeren set sayısı: 24


,set_num,name,year
13041,53894-1,Watercolor Set,2027
13042,53895-1,Duplo Water Art Play Mat,2027
27238,LGLKE262H-1,Rechargeable Minifigure Key Light (All Colors),2027
25054,9781534357921-1,Ninjago: Masters of Spinjitzu,2027
24517,9780241838570-1,City: Game On: Penguin Peril,2027
24516,9780241838563-1,Ninjago: Dragons Rising: Riyu the Dragon: Nood...,2027
24515,9780241838389-1,DK Super Readers Level 1: Ninjago: Go Team Ninja,2027
24514,9780241836712-1,DK Super Readers Level 2: Marvel Super Heroes:...,2027
24513,9780241831953-1,Minecraft: Would you Rather?,2027
19537,757894517472-1,Chequered Brick Lunch Bag,2027


In [10]:
sets["year"].describe()


count    28180.000000
mean      2010.979915
std         14.031003
min       1949.000000
25%       2004.000000
50%       2015.000000
75%       2021.000000
max       2027.000000
Name: year, dtype: float64

## 5. `sets.theme_id` → `themes.id` join kontrolü

Her set'in theme_id değeri themes tablosunda karşılık bulmalı; null theme_id ve "yetim" (orphan) theme_id'leri ayrı ayrı raporlanıyor.


In [11]:
null_theme_id = sets["theme_id"].isna().sum()
print(f"Null theme_id sayısı: {null_theme_id} ({null_theme_id / len(sets) * 100:.2f}%)")

valid_theme_ids = set(themes["id"])
non_null_theme_ids = sets["theme_id"].dropna()
orphan_mask = ~non_null_theme_ids.isin(valid_theme_ids)
orphan_theme_ids = non_null_theme_ids[orphan_mask]

print(f"themes.id'de karşılığı olmayan (orphan) theme_id sayısı: {orphan_mask.sum()} "
      f"({orphan_mask.sum() / len(sets) * 100:.2f}%)")

if orphan_mask.sum():
    print("Orphan theme_id örnekleri:")
    display(sets.loc[orphan_theme_ids[orphan_mask].index, ["set_num", "name", "theme_id"]].head(10))
else:
    print("✅ Tüm non-null theme_id değerleri themes.id içinde mevcut — join sorunsuz.")


Null theme_id sayısı: 0 (0.00%)
themes.id'de karşılığı olmayan (orphan) theme_id sayısı: 0 (0.00%)
✅ Tüm non-null theme_id değerleri themes.id içinde mevcut — join sorunsuz.


## 6. `num_parts = 0` olan setler

Bu setler muhtemelen minifig-only setler, promosyon (gift-with-purchase) ürünleri veya parça sayısı hiç girilmemiş kayıtlar.
Ana analiz (ör. set büyüklüğü / karmaşıklık trendleri) bu setlerle çarpıtılacağından, ayrı ayrı raporlanıp sonraki adımlarda filtrelenmesi öneriliyor.


In [12]:
zero_parts = sets[sets["num_parts"] == 0]
zero_ratio = len(zero_parts) / len(sets) * 100

print(f"num_parts = 0 olan set sayısı: {len(zero_parts):,} / {len(sets):,}")
print(f"Oran: {zero_ratio:.2f}%")


num_parts = 0 olan set sayısı: 8,138 / 28,180
Oran: 28.88%


In [13]:
zero_parts_with_theme = zero_parts.merge(
    themes[["id", "name"]].rename(columns={"id": "theme_id", "name": "theme_name"}),
    on="theme_id", how="left",
)

print("num_parts=0 setlerin en sık geçtiği temalar (ilk 15):")
zero_parts_with_theme["theme_name"].value_counts().head(15)


num_parts=0 setlerin en sık geçtiği temalar (ilk 15):


theme_name
Bags, Totes, & Luggage            973
Key Chain                         785
Clothing & Footwear               704
Stationery and Office Supplies    674
Houseware                         614
Gear                              507
Story Books                       385
Video Games and Accessories       277
Activity Books                    253
Role Play Toys and Costumes       235
Plush Toys                        162
Storage                           138
Non-fiction Books                 130
Bag and Luggage Tags              130
Clocks and Watches                111
Name: count, dtype: int64

In [14]:
zero_parts.sample(min(10, len(zero_parts)), random_state=42)[["set_num", "name", "year", "theme_id", "num_parts"]]


,set_num,name,year,theme_id,num_parts
3106,203062508-1,Ninjago Lloyd Backpack with Gym Bag and Pencil...,2026,777,0
15763,6572607-1,Easter Stickers,2025,501,0
7250,4060-2,Multi Basket,2014,740,0
3843,238-3,LEGO System Idea Book no. 1,1962,757,0
7470,4084039-1,Marvel Avengers with Silver Centurion Minifigu...,2016,742,0
27418,LMP301A-1,Star Wars: Darth Vader on the Rebel Hunt,2019,759,0
15950,66151-1,Limited Edition Green Brick Tub Value-Pack,2006,505,0
24600,9780545649926-1,Legends of Chima: How to Draw: Heroes and Vill...,2014,760,0
12166,5010250-1,Nike x LEGO Collection Big Kids' T-Shirt - White,2026,799,0
21319,850451-1,Lord Vampyre Key Chain,2012,503,0


## 7. Özet

- **sets.csv**: satır/kolon sayısı, dtype'lar ve null oranları yukarıda raporlandı.
- **themes.csv**: satır/kolon sayısı, dtype'lar ve null oranları yukarıda raporlandı.
- **Yıl aralığı**: `sets.year` min–max değerleri ve bugünün yılından büyük anormal kayıtlar kontrol edildi.
- **Join kontrolü**: `sets.theme_id` → `themes.id` eşleşmesinde null ve orphan (eşleşmeyen) kayıt sayıları raporlandı.
- **`num_parts = 0`**: Bu setlerin oranı hesaplandı; bunlar muhtemelen minifig-only / promosyon setleri olduğundan **ana analizden (set büyüklüğü, parça trendleri vb.) filtrelenmesi öneriliyor.**
